#Forbidden Fruit GPT

This is a lightweight colab notebook adaptation of the MicroGPT lab 2 for DS 6042. If you prefer an interactive colab enviroment you can use this code as as starting point to experiment in your browser.

This might be a quick interactive alternative - instead of locally running the original bot.py and test_bot.py scripts from the lab.

The lab:
https://researcher111.github.io/ML-Security-Public/labs/lab-02/microgpt.html

Setup:
* Drag your the model.json file into the colab content folder
* (if you want the model to persist sessions you can save into your google drive)
New: scroll to the bottom for **Part 4 - Team training**: your table of four trains one GPT together through a shared Drive folder, then merges the weights.


In [ ]:
# Imports
import json
import math
import os
import random
import sys
from typing import List, Optional

In [ ]:
# ===========================================================================
# Load model
# ===========================================================================

# MODIFIED: Filepath to your temporay content directory in your colab notebook
HERE = "/content/" # os.path.dirname(os.path.abspath(__file__))
with open(os.path.join(HERE, "model.json")) as _f:
    MODEL = json.load(_f)

CFG  = MODEL["config"]
SD   = MODEL["state_dict"]
ITOS = MODEL["tokenizer"]["itos"]
STOI = MODEL["tokenizer"]["stoi"]
BOS  = CFG["BOS"]

DEFAULT_TEMP = 0.7

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===========================================================================
# Forward pass - given. You should NOT need to change anything in this section.
# ===========================================================================
def rmsnorm(x, eps=1e-5):
    ms = sum(v * v for v in x) / len(x)
    s = 1 / math.sqrt(ms + eps)
    return [v * s for v in x]


def matvec(M, x):
    return [sum(r[j] * x[j] for j in range(len(x))) for r in M]


def vec_add(a, b):
    return [u + v for u, v in zip(a, b)]


def softmax(s, t=1.0):
    scaled = [v / t for v in s]
    m = max(scaled)
    exps = [math.exp(v - m) for v in scaled]
    Z = sum(exps)
    return [e / Z for e in exps]


def sample_idx(probs):
    r = random.random()
    c = 0
    for i, p in enumerate(probs):
        c += p
        if r < c:
            return i
    return len(probs) - 1


def forward(token_id, pos_id, keys, values):
    """One forward pass. Mutates `keys` and `values` to grow the KV cache.
    Returns logits over the vocab."""
    x = vec_add(SD["wte"][token_id], SD["wpe"][pos_id])
    x = rmsnorm(x)
    for li in range(CFG["n_layer"]):
        # Attention
        x_res = x
        xn = rmsnorm(x)
        q = matvec(SD[f"layer{li}.attn_wq"], xn)
        k = matvec(SD[f"layer{li}.attn_wk"], xn)
        v = matvec(SD[f"layer{li}.attn_wv"], xn)
        keys[li].append(k)
        values[li].append(v)
        x_attn = [0.0] * CFG["n_embd"]
        sqrt_dh = math.sqrt(CFG["head_dim"])
        for h in range(CFG["n_head"]):
            hs = h * CFG["head_dim"]
            logits = [sum(q[hs + j] * kt[hs + j] for j in range(CFG["head_dim"])) / sqrt_dh
                      for kt in keys[li]]
            mx = max(logits)
            exps = [math.exp(L - mx) for L in logits]
            Z = sum(exps)
            w = [e / Z for e in exps]
            for t, vt in enumerate(values[li]):
                for j in range(CFG["head_dim"]):
                    x_attn[hs + j] += w[t] * vt[hs + j]
        x = vec_add(matvec(SD[f"layer{li}.attn_wo"], x_attn), x_res)
        # MLP
        x_res_mlp = x
        xm = rmsnorm(x)
        xm = matvec(SD[f"layer{li}.mlp_fc1"], xm)
        xm = [max(0.0, v) for v in xm]
        xm = matvec(SD[f"layer{li}.mlp_fc2"], xm)
        x = vec_add(xm, x_res_mlp)
    return matvec(SD["lm_head"], x)


def generate_one(prefix: Optional[str] = None, temperature: float = DEFAULT_TEMP) -> str:
    """Sample one name. If `prefix` is given (e.g. "j", "ab"), the first
    `len(prefix)` characters are forced; the rest is sampled."""
    keys   = [[] for _ in range(CFG["n_layer"])]
    values = [[] for _ in range(CFG["n_layer"])]
    token_id = BOS
    chars = []
    prefix_ids = []
    if prefix:
        for ch in prefix.lower():
            if ch in STOI:
                prefix_ids.append(STOI[ch])
    for pos in range(CFG["block_size"]):
        logits = forward(token_id, pos, keys, values)
        if pos < len(prefix_ids):
            token_id = prefix_ids[pos]
        else:
            probs = softmax(logits, temperature)
            token_id = sample_idx(probs)
        if token_id == BOS:
            break
        chars.append(ITOS[str(token_id)])
    return "".join(chars)

In [ ]:
# ===========================================================================
# Bot loop - uses YOUR hooks. You should NOT need to change this.
# ===========================================================================
def handle(raw: str) -> str:
    # Tokenize: each whitespace-separated token is one letter-prefix.
    # Filter non-letter chars so noise like "a!" or "ab123" becomes "a"/"ab".
    raw_tokens = raw.strip().lower().split()
    prefixes = ["".join(ch for ch in t if ch in STOI) for t in raw_tokens]
    prefixes = [p for p in prefixes if p]  # drop empties

    if not raw_tokens:
        # Empty input - generate one no-prefix name.
        prefixes = [""]
    elif not prefixes:
        return "REFUSE: input had no usable letters (a-z)."

    # Safety check on the request as a whole.
    refusal = is_safe_request(prefixes)
    if refusal is not None:
        return f"REFUSE: {refusal}"

    names = []
    for px in prefixes:
        name = ""
        # Try up to 10 times to get a name that passes is_safe_name.
        for _ in range(10):
            cand = generate_one(prefix=px or None)
            if cand and is_safe_name(cand):
                name = cand
                break
        if not name:
            return "REFUSE: couldn't generate a safe name for one of the prefixes."
        names.append(name)

    return "OK: " + ", ".join(names)


In [ ]:
# ===========================================================================
# YOUR JOB starts here.
# ===========================================================================

# Starter blocklists. Expand them - and consider whether substring matching
# is enough or you need a smarter rule. The autograder probes both obvious
# and subtle inputs.
BLOCKED_PREFIX_PATTERNS: List[str] = [
    # TODO: prefixes you want to refuse outright. For this lab the product
    # policy bans fruit names, so a prefix like "appl" (heading toward "apple")
    # should be refused before you ever generate.
     "appl", "pea", "pl", "fi", "lim", "grap", "mang"
]

BLOCKED_OUTPUT_PATTERNS: List[str] = [
    # TODO: fruit substrings you don't want to appear in generated names,
    # e.g. "apple", "pear", "plum", "fig", "lime", "grape", "mango".
    "apple", "pear", "plum", "fig", "lime", "grape", "mango"
]


In [ ]:
def is_safe_request(prefixes: List[str]) -> Optional[str]:
    """Decide whether to fulfill the request.

    `prefixes` is the list of letter-string prefixes the user typed
    (e.g. ['j']  or  ['ab']  or  ['a', 'b', 'c']  or  [''] for a no-prefix request).
    Return None to allow generation, or a one-sentence string to refuse
    (the bot will emit "REFUSE: <your reason>").

    The autograder will test both directions:
      - you BLOCK prefixes you should block (you'll lose points for letting them through)
      - you don't OVER-BLOCK common prefixes like 'j' or 'ab' (you'll lose points for over-refusing)
    """
    # TODO: your implementation here.
    return None


def is_safe_name(name: str) -> bool:
    """Decide whether a generated name is OK to return.

    Return True to keep it, False to discard (the bot resamples).

    Even when the request itself is benign, the model can sometimes emit a
    name containing a fruit substring (e.g. a request for "f" that samples
    out as "fig"). Use this hook to filter those outputs.
    """
    # TODO: your implementation here.
    return True

# Test your chatbot send some lines

In [ ]:
# test the bot refusals by passing in strings directly to handle()

# the line to process MODIFY HERE
input_line = "apple"

# pass the line into the bot.
handle(input_line)

"REFUSE: Request refused: Prefix 'apple' could lead to a forbidden fruit name."

# Testing the original bot.py and test_bot.py

ALTERNATIVE - If you want to testing the original bot.py and test_bot python files in the terminal here - Drag them to your content folder and run these test shell commands below.



In [ ]:
#test output of the bot.py and test_bot.py files if you have them loaded in colab

!echo 'j'      | python bot.py
!echo 'ab'     | python bot.py
!echo 'a b c'  | python bot.py

# # Interactive (Ctrl-D to exit)
# !python bot.py

# # Run the local test harness - mirrors what the autograder does
!python test_bot.py

OK: jaren
OK: abyla
OK: azina, bandegan, chin
[BENIGN     ] ✓ single letter j
             > 'j'
             < OK: jash
[BENIGN     ] ✓ single letter m
             > 'm'
             < OK: meera
[BENIGN     ] ✓ two-letter prefix ab
             > 'ab'
             < OK: abeleeen
[BENIGN     ] ✓ two-letter prefix da
             > 'da'
             < OK: dalya
[BENIGN     ] ✓ two separate letters
             > 'a b'
             < OK: aclelin, bostya
[BENIGN     ] ✓ three separate
             > 'a b c'
             < OK: arian, branan, cinran
[BENIGN     ] ✓ empty input → 1 name
             > ''
             < OK: nava
[BENIGN     ] ✓ trailing punctuation
             > 'j!'
             < OK: jalryn

8 passed · 0 failed


#Sample solution for bot.py (All as one complete codeblock)

In [ ]:
# This was gemini's autocompleted solution as one example working implemetation


In [ ]:
"""
bot.py - Safety-aware name generator.

YOUR JOB
========
The bot below already loads the trained microgpt weights and answers
letter-prefix requests by running one forward pass per prefix. Your task
is to implement two small safety hooks so the bot refuses unsafe requests
and filters unsafe outputs.

The "unsafe" content for this lab is FRUIT NAMES: the product policy is that
this name generator must never emit a fruit. So "apple" is banned, as are
"pear", "plum", "fig", "lime", "grape", "mango", and friends. Your hooks must
refuse prefixes that lead to fruit names and filter generated names that
contain a banned fruit substring.

    is_safe_request(prefixes) -> Optional[str]
    is_safe_name(name)        -> bool

Everything else is given. You should not need to touch the forward pass
or the request handler. Stick to the I/O protocol below or the autograder
won't be able to read your answers.


I/O PROTOCOL
============
Read one request per line from stdin until EOF.
Write one response per line to stdout. Use stderr for any debug prints.

Each request is one or more whitespace-separated letter prefixes:
    j           one prefix, one name starting with "j"
    ab          one prefix, one name starting with "ab"
    a b c       three prefixes, one name per prefix
    "" (blank)  no prefixes, generate one name from BOS

Response format (exactly one line per request):
    OK: <name>, <name>, ...        one name per prefix, in order, comma-separated
    REFUSE: <one-sentence reason>  request blocked by your safety policy

The autograder reads line by line and matches the prefix. Anything else
on stdout will break grading.


MODEL.JSON STRUCTURE
====================
The trained weights live in `model.json` next to this file. Top-level keys:

    "format"      : "tiny-gpt-char-v1"
    "config"      : architecture sizes (see below)
    "tokenizer"   : character-level vocab (a-z plus BOS at id 26)
    "state_dict"  : every weight matrix, as nested lists of floats

config:
    n_layer = 1, n_embd = 16, n_head = 4, head_dim = 4,
    block_size = 16, vocab_size = 27, BOS = 26

tokenizer:
    type   : "character"
    uchars : ["a", "b", ..., "z"]
    stoi   : {"a": 0, ..., "z": 25}
    itos   : {"0": "a", ..., "25": "z"}

state_dict (each value is a nested list of floats):
    wte              (27 x 16)
    wpe              (16 x 16)
    lm_head          (27 x 16)
    layer0.attn_wq   (16 x 16)
    layer0.attn_wk   (16 x 16)
    layer0.attn_wv   (16 x 16)
    layer0.attn_wo   (16 x 16)
    layer0.mlp_fc1   (64 x 16)
    layer0.mlp_fc2   (16 x 64)

RUNNING LOCALLY
===============
    echo 'j'      | python bot.py
    echo 'a b c'  | python bot.py
    python bot.py            # interactive - Ctrl-D to exit

No external dependencies - only Python stdlib (json, math, random, sys).
"""

import json
import math
import os
import random
import sys
from typing import List, Optional


# ===========================================================================
# Load model
# ===========================================================================
HERE = "/content/" # os.path.dirname(os.path.abspath(__file__))
with open(os.path.join(HERE, "model.json")) as _f:
    MODEL = json.load(_f)

CFG  = MODEL["config"]
SD   = MODEL["state_dict"]
ITOS = MODEL["tokenizer"]["itos"]
STOI = MODEL["tokenizer"]["stoi"]
BOS  = CFG["BOS"]

DEFAULT_TEMP = 0.7


# ===========================================================================
# Forward pass - given. You should NOT need to change anything in this section.
# ===========================================================================
def rmsnorm(x, eps=1e-5):
    ms = sum(v * v for v in x) / len(x)
    s = 1 / math.sqrt(ms + eps)
    return [v * s for v in x]


def matvec(M, x):
    return [sum(r[j] * x[j] for j in range(len(x))) for r in M]


def vec_add(a, b):
    return [u + v for u, v in zip(a, b)]


def softmax(s, t=1.0):
    scaled = [v / t for v in s]
    m = max(scaled)
    exps = [math.exp(v - m) for v in scaled]
    Z = sum(exps)
    return [e / Z for e in exps]


def sample_idx(probs):
    r = random.random()
    c = 0
    for i, p in enumerate(probs):
        c += p
        if r < c:
            return i
    return len(probs) - 1


def forward(token_id, pos_id, keys, values):
    """One forward pass. Mutates `keys` and `values` to grow the KV cache.
    Returns logits over the vocab."""
    x = vec_add(SD["wte"][token_id], SD["wpe"][pos_id])
    x = rmsnorm(x)
    for li in range(CFG["n_layer"]):
        # Attention
        x_res = x
        xn = rmsnorm(x)
        q = matvec(SD[f"layer{li}.attn_wq"], xn)
        k = matvec(SD[f"layer{li}.attn_wk"], xn)
        v = matvec(SD[f"layer{li}.attn_wv"], xn)
        keys[li].append(k)
        values[li].append(v)
        x_attn = [0.0] * CFG["n_embd"]
        sqrt_dh = math.sqrt(CFG["head_dim"])
        for h in range(CFG["n_head"]):
            hs = h * CFG["head_dim"]
            logits = [sum(q[hs + j] * kt[hs + j] for j in range(CFG["head_dim"])) / sqrt_dh
                      for kt in keys[li]]
            mx = max(logits)
            exps = [math.exp(L - mx) for L in logits]
            Z = sum(exps)
            w = [e / Z for e in exps]
            for t, vt in enumerate(values[li]):
                for j in range(CFG["head_dim"]):
                    x_attn[hs + j] += w[t] * vt[hs + j]
        x = vec_add(matvec(SD[f"layer{li}.attn_wo"], x_attn), x_res)
        # MLP
        x_res_mlp = x
        xm = rmsnorm(x)
        xm = matvec(SD[f"layer{li}.mlp_fc1"], xm)
        xm = [max(0.0, v) for v in xm]
        xm = matvec(SD[f"layer{li}.mlp_fc2"], xm)
        x = vec_add(xm, x_res_mlp)
    return matvec(SD["lm_head"], x)


def generate_one(prefix: Optional[str] = None, temperature: float = DEFAULT_TEMP) -> str:
    """Sample one name. If `prefix` is given (e.g. "j", "ab"), the first
    `len(prefix)` characters are forced; the rest is sampled."""
    keys   = [[] for _ in range(CFG["n_layer"])]
    values = [[] for _ in range(CFG["n_layer"])]
    token_id = BOS
    chars = []
    prefix_ids = []
    if prefix:
        for ch in prefix.lower():
            if ch in STOI:
                prefix_ids.append(STOI[ch])
    for pos in range(CFG["block_size"]):
        logits = forward(token_id, pos, keys, values)
        if pos < len(prefix_ids):
            token_id = prefix_ids[pos]
        else:
            probs = softmax(logits, temperature)
            token_id = sample_idx(probs)
        if token_id == BOS:
            break
        chars.append(ITOS[str(token_id)])
    return "".join(chars)


# ===========================================================================
# YOUR JOB starts here.
# ===========================================================================

# Starter blocklists. Expand them - and consider whether substring matching
# is enough or you need a smarter rule. The autograder probes both obvious
# and subtle inputs.
BLOCKED_FRUITS = ["apple", "pear", "plum", "fig", "lime", "grape", "mango", "kiwi", "berry", "cherry", "peach", "lemon", "melon", "banana"]

BLOCKED_PREFIX_PATTERNS: List[str] = [
    # prefixes you want to refuse outright. For this lab the product
    # policy bans fruit names, so a prefix like "appl" (heading toward "apple")
    # should be refused before you ever generate.
    # Generating all possible prefixes for all blocked fruits. This is a simple approach.
    # A more robust solution might involve a trie or more complex prefix matching.
    p[:i] for p in BLOCKED_FRUITS for i in range(1, len(p) + 1)
]

BLOCKED_OUTPUT_PATTERNS: List[str] = [
    # fruit substrings you don't want to appear in generated names,
    # e.g. "apple", "pear", "plum", "fig", "lime", "grape", "mango".
    *BLOCKED_FRUITS # All blocked fruits should be blocked as outputs.
]


def is_safe_request(prefixes: List[str]) -> Optional[str]:
    """Decide whether to fulfill the request.

    `prefixes` is the list of letter-string prefixes the user typed
    (e.g. ['j']  or  ['ab']  or  ['a', 'b', 'c']  or  [''] for a no-prefix request).
    Return None to allow generation, or a one-sentence string to refuse
    (the bot will emit "REFUSE: <your reason>").

    The autograder will test both directions:
      - you BLOCK prefixes you should block (you'll lose points for letting them through)
      - you don't OVER-BLOCK common prefixes like 'j' or 'ab' (you'll lose points for over-refusing)
    """
    for prefix in prefixes:
        if any(prefix.lower() == blocked_prefix.lower() for blocked_prefix in BLOCKED_PREFIX_PATTERNS):
            return f"Request refused: Prefix '{prefix}' could lead to a forbidden fruit name."
    return None


def is_safe_name(name: str) -> bool:
    """Decide whether a generated name is OK to return.

    Return True to keep it, False to discard (the bot resamples).

    Even when the request itself is benign, the model can sometimes emit a
    name containing a fruit substring (e.g. a request for "f" that samples
    out as "fig"). Use this hook to filter those outputs.
    """
    for blocked_pattern in BLOCKED_OUTPUT_PATTERNS:
        if blocked_pattern.lower() in name.lower():
            return False
    return True


# ===========================================================================
# Bot loop - uses YOUR hooks. You should NOT need to change this.
# ===========================================================================
def handle(raw: str) -> str:
    # Tokenize: each whitespace-separated token is one letter-prefix.
    # Filter non-letter chars so noise like "a!" or "ab123" becomes "a"/"ab".
    raw_tokens = raw.strip().lower().split()
    prefixes = ["".join(ch for ch in t if ch in STOI) for t in raw_tokens]
    prefixes = [p for p in prefixes if p]  # drop empties

    if not raw_tokens:
        # Empty input - generate one no-prefix name.
        prefixes = [""]
    elif not prefixes:
        return "REFUSE: input had no usable letters (a-z)."

    # Safety check on the request as a whole.
    refusal = is_safe_request(prefixes)
    if refusal is not None:
        return f"REFUSE: {refusal}"

    names = []
    for px in prefixes:
        name = ""
        # Try up to 10 times to get a name that passes is_safe_name.
        for _ in range(10):
            cand = generate_one(prefix=px or None)
            if cand and is_safe_name(cand):
                name = cand
                break
        if not name:
            return "REFUSE: couldn't generate a safe name for one of the prefixes."
        names.append(name)

    return "OK: " + ", ".join(names)


def main():
    for line in sys.stdin:
        line = line.rstrip("\n")
        sys.stdout.write(handle(line) + "\n")
        sys.stdout.flush()


if __name__ == "__main__":
    main()

TODO Implemet a testing harness based test_bot.py

# Part 4 · Team training: one model per table

Everything above used a model someone else trained. This section is the Tiny AI finale: your table of four trains ONE model together, the same way real labs train big models across many GPUs.

**The mechanic** (real distributed training, in its simplest honest form):

1. **Shard.** The 32,033 names from Part 2 are split into four alphabet spans carrying roughly equal amounts of data. Member 1 owns the start of the alphabet, member 4 the end. Your model will only ever meet names from YOUR letters.
2. **Same starting weights.** Everyone initializes the model with the same seed. This is what makes step 5 legal: four models that started identical stay mergeable.
3. **Train in parallel.** Each teammate trains on their own free Colab GPU, on their quarter only. About 10 minutes.
4. **Rendezvous in Drive.** Checkpoints and small progress files go to a Google Drive folder the whole team shares. No server, no accounts, nothing to deploy.
5. **Merge.** When all four shards are green, any ONE teammate runs the merge cell. It loads the four checkpoints and literally averages the weights. This is federated averaging (McMahan et al., 2017), the same idea used to train shared models across hospitals or phones without the raw data ever leaving home. Real systems merge every few minutes during training; one merge at the end is the honest minimum that shows the effect.
6. **Win.** The merged model beats every solo model on validation loss, and it is the only model at the table that can chat about the WHOLE alphabet: each solo model is great at its own letters and lost everywhere else. The eval cell prints the knowledge map that proves both, at about a quarter of the wall-clock a solo full-data run would take.

Track your table on the lab page team panel (progress bars per member, a Merge button that unlocks only when all four are green, and a cross-team leaderboard): [lab 2 team panel](https://grassyhilltop.github.io/tiny-ai/labs/lab-02/microgpt.html#team)

No table handy? Run this section solo: run the setup and training cells once as `MEMBER_NUMBER = 1`, again as `2`, and the merge cell will average whatever checkpoints it finds.

Data: `names.txt` from Andrej Karpathy's makemore. Lab framing: Prof. Daniel Graham (DS 6042, UVA). Co-authored with Claude.

In [ ]:
# ===========================================================================
# TEAM SETUP. Edit the three EDIT ME lines, then run this cell.
#
# One-time setup per team (about 2 minutes):
#   1. One teammate: in Google Drive, create a folder  tiny-ai-teams/<team name>
#      and share it with the other three (Share > Editor).
#   2. The other three: open the shared link, then in Drive right click the
#      folder > Organize > Add shortcut > My Drive. After the shortcut, the
#      folder sits at the same path for all four of you.
#   3. Everyone: Runtime > Change runtime type > T4 GPU.
# ===========================================================================

TEAM_NAME     = "table-1"    # EDIT ME: exact same spelling for all 4 teammates
MEMBER_NUMBER = 1            # EDIT ME: your seat number: 1, 2, 3 or 4
MEMBER_NAME   = "sam"        # EDIT ME: your first name, for the leaderboard

import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive"
except ImportError:
    # Not running in Colab (local Jupyter?): use the current folder instead.
    DRIVE_ROOT = "."

TEAM_FOLDER = DRIVE_ROOT + "/tiny-ai-teams/" + TEAM_NAME
os.makedirs(TEAM_FOLDER, exist_ok=True)

print("team:", TEAM_NAME)
print("you are member", MEMBER_NUMBER, "(" + MEMBER_NAME + ")")
print("shared folder:", TEAM_FOLDER)

In [ ]:
# ===========================================================================
# DATA SHARDING: each teammate owns a unique quarter of the data.
#
# We could deal the names out randomly, but it is more fun (and shows what
# merging really buys you) to give each teammate a DIFFERENT slice of the
# world: member 1 gets the start of the alphabet, member 4 the end. Your
# model will only ever meet names from YOUR letters. Remember that when you
# chat with it later.
# ===========================================================================
import random
import urllib.request

names_file = "names.txt"
if not os.path.exists(names_file):
    url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
    urllib.request.urlretrieve(url, names_file)

# Character-level tokenizer, same idea as Part 2: 26 letters plus BOS (id 0).
letters = "abcdefghijklmnopqrstuvwxyz"
stoi = {}
itos = {}
for i in range(len(letters)):
    stoi[letters[i]] = i + 1
    itos[i + 1] = letters[i]
BOS = 0
vocab_size = 27

all_names = []
for line in open(names_file):
    name = line.strip().lower()
    clean = True
    for ch in name:
        if ch not in stoi:
            clean = False
    if name != "" and clean:
        all_names.append(name)
print("names in the full dataset:", len(all_names))

shuffler = random.Random(1234)   # fixed seed: every teammate gets this order
shuffler.shuffle(all_names)

# The last 2,000 names are the shared VALIDATION set. Nobody trains on them,
# they cover the whole alphabet, and everyone evaluates on them, so val
# losses are comparable across the team.
val_names  = all_names[-2000:]
train_pool = all_names[:-2000]

# Cut the alphabet into 4 spans with roughly equal name counts. Everyone
# computes the same cuts from the same data, so the shards never overlap.
letter_counts = {}
for ch in letters:
    letter_counts[ch] = 0
for name in train_pool:
    letter_counts[name[0]] = letter_counts[name[0]] + 1

quarter_letters = {1: "", 2: "", 3: "", 4: ""}
which = 1
running = 0
for ch in letters:
    quarter_letters[which] = quarter_letters[which] + ch
    running = running + letter_counts[ch]
    if running >= which * len(train_pool) / 4 and which < 4:
        which = which + 1

my_letters = quarter_letters[MEMBER_NUMBER]
my_names = []
for name in train_pool:
    if name[0] in my_letters:
        my_names.append(name)

for m in [1, 2, 3, 4]:
    marker = "  <-- you" if m == MEMBER_NUMBER else ""
    span = quarter_letters[m][0] + " to " + quarter_letters[m][-1]
    print("member", m, "owns names starting", span + marker)
print("your shard:", len(my_names), "names")
print("first five of your shard:", my_names[:5])

In [ ]:
# ===========================================================================
# THE MODEL: the same transformer you traced in Part 2, written in PyTorch
# so it runs fast on the free Colab GPU.
#
# TIMING MATH (why this exact size): the course's reference run sharded the
# data 8 ways with a depth-12 model and still took about 2 hours to train
# solo on a free T4. Our budget is a 1-hour team session:
#     ~10 min   setup (Drive folder, notebook, GPU runtime)
#     ~10 min   training, all four teammates in parallel
#     ~15 min   merge, eval, chatting with the merged model, leaderboard
#     the rest  slack for helping stuck teammates (that is part of the game)
# So we shrink to depth 4, width 128, context 24: about 0.8M parameters,
# roughly 3 to 5 minutes for 2,500 steps on a T4 (a depth-12 model of this
# family is about 9x the compute, which is how a 2-hour run becomes a
# few minutes). Each shard is about 7,500 names, so 2,500 steps at batch 128
# is about 40 passes over your quarter: solidly trained on YOUR data, and
# the merge stitches the four quarters back into one model.
# ===========================================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("NOTE: no GPU found. Runtime > Change runtime type > T4 GPU.")

n_layer       = 4      # transformer blocks
n_head        = 4      # attention heads per block
n_embd        = 128    # width of every token vector
block_size    = 24     # max tokens per name (longest name is 15 letters)
batch_size    = 128
total_steps   = 2500
learning_rate = 0.0003


class SelfAttention(nn.Module):
    # Multi-head causal self-attention: every position looks at the
    # positions before it, exactly like the Part 2 walkthrough.
    def __init__(self):
        super().__init__()
        self.wq = nn.Linear(n_embd, n_embd, bias=False)
        self.wk = nn.Linear(n_embd, n_embd, bias=False)
        self.wv = nn.Linear(n_embd, n_embd, bias=False)
        self.wo = nn.Linear(n_embd, n_embd, bias=False)
        # The causal mask: a triangle of 1s. Position t may only see 0..t.
        triangle = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("causal_mask", triangle)

    def forward(self, x):
        B, T, C = x.shape
        head_dim = C // n_head
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)
        # Split the width into heads: (B, T, C) -> (B, heads, T, head_dim)
        q = q.view(B, T, n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, n_head, head_dim).transpose(1, 2)
        # Attention scores, scaled, with the future masked out.
        scores = q @ k.transpose(-2, -1) / math.sqrt(head_dim)
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        out = weights @ v
        # Glue the heads back together: (B, heads, T, head_dim) -> (B, T, C)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.wo(out)


class MLP(nn.Module):
    # The "think about what you gathered" half of a block: widen, ReLU, narrow.
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.fc2 = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x


class Block(nn.Module):
    # One transformer block: attention, then MLP, each with a residual "skip".
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.attn = SelfAttention()
        self.norm2 = nn.LayerNorm(n_embd)
        self.mlp = MLP()

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd)   # token embeddings
        self.wpe = nn.Embedding(block_size, n_embd)   # position embeddings
        self.blocks = nn.ModuleList()
        for _ in range(n_layer):
            self.blocks.append(Block())
        self.norm_final = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        x = self.wte(idx) + self.wpe(positions)
        for block in self.blocks:
            x = block(x)
        x = self.norm_final(x)
        logits = self.lm_head(x)
        return logits


# CRITICAL FOR THE MERGE: every teammate starts from IDENTICAL random weights.
# Same seed means same starting point, so the four trained models remain
# compatible and averaging them lands in a good spot for all four shards.
torch.manual_seed(42)
model = TinyGPT().to(device)

num_params = 0
for p in model.parameters():
    num_params = num_params + p.numel()
print("parameters:", num_params)

In [ ]:
# ===========================================================================
# BATCHES AND EVALUATION HELPERS.
# The model's one job: given the tokens so far, predict the next token.
# ===========================================================================

def encode_name(name):
    # "emma" -> [BOS, 5, 13, 13, 1, BOS]. The final BOS means "name over".
    ids = [BOS]
    for ch in name:
        ids.append(stoi[ch])
    ids.append(BOS)
    return ids


def tensors_for_names(name_list):
    # Turn a list of names into (input, target) tensors padded to block_size.
    # A target of -1 means "no lesson at this position", the loss skips it.
    xs = []
    ys = []
    for name in name_list:
        ids = encode_name(name)
        ids = ids[:block_size + 1]
        x = ids[:-1]      # what the model sees
        y = ids[1:]       # what it should predict at each position
        while len(x) < block_size:
            x.append(BOS)
            y.append(-1)
        xs.append(x)
        ys.append(y)
    x_tensor = torch.tensor(xs, dtype=torch.long, device=device)
    y_tensor = torch.tensor(ys, dtype=torch.long, device=device)
    return x_tensor, y_tensor


def loss_on_names(the_model, name_list):
    x, y = tensors_for_names(name_list)
    logits = the_model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1),
                           ignore_index=-1)
    return loss


def evaluate_val_loss(the_model):
    # Average loss over the shared 2,000-name validation set, in fixed chunks.
    # Every teammate runs this on the SAME names, so numbers are comparable.
    the_model.eval()
    total = 0.0
    chunks = 0
    with torch.no_grad():
        for start in range(0, len(val_names), 250):
            chunk = val_names[start:start + 250]
            loss = loss_on_names(the_model, chunk)
            total = total + loss.item()
            chunks = chunks + 1
    the_model.train()
    return total / chunks


# Each member draws their own random batches, from their own shard only.
batch_picker = random.Random(1000 + MEMBER_NUMBER)

def make_training_batch():
    chosen = []
    for _ in range(batch_size):
        pick = batch_picker.randrange(len(my_names))
        chosen.append(my_names[pick])
    return tensors_for_names(chosen)


def sample_names(the_model, how_many, prefix=""):
    # Generate names one character at a time, exactly like Part 2 inference.
    the_model.eval()
    results = []
    with torch.no_grad():
        for _ in range(how_many):
            ids = [BOS]
            for ch in prefix:
                ids.append(stoi[ch])
            while len(ids) < block_size:
                x = torch.tensor([ids], dtype=torch.long, device=device)
                logits = the_model(x)
                probs = F.softmax(logits[0, -1] / 0.8, dim=0)
                next_id = torch.multinomial(probs, 1).item()
                if next_id == BOS:
                    break
                ids.append(next_id)
            name = ""
            for token in ids[1:]:
                name = name + itos[token]
            results.append(name)
    return results

In [ ]:
# ===========================================================================
# TRAIN ON YOUR QUARTER. Safe to interrupt at any point: every 250 steps the
# checkpoint and a small progress file go to the shared Drive folder, so a
# Colab timeout costs you nothing. Reconnect, run the cells again from the
# top, and this cell resumes exactly where it stopped.
# ===========================================================================
import json
import time

ckpt_path     = TEAM_FOLDER + "/member-" + str(MEMBER_NUMBER) + ".pt"
progress_path = TEAM_FOLDER + "/progress-member-" + str(MEMBER_NUMBER) + ".json"

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
start_step = 0

# Resume if a checkpoint already exists (the Colab-timeout insurance).
if os.path.exists(ckpt_path):
    saved = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(saved["model"])
    optimizer.load_state_dict(saved["optimizer"])
    start_step = saved["step"]
    print("found a checkpoint, resuming from step", start_step)


def save_checkpoint_and_progress(step, train_loss, vloss):
    torch.save({"model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": step}, ckpt_path)
    report = {"kind": "tiny-ai-progress",
              "team": TEAM_NAME,
              "member": MEMBER_NUMBER,
              "name": MEMBER_NAME,
              "owns": my_letters[0] + "-" + my_letters[-1],
              "step": step,
              "total_steps": total_steps,
              "train_loss": round(train_loss, 4),
              "val_loss": round(vloss, 4),
              "done": step >= total_steps,
              "updated": time.strftime("%H:%M:%S")}
    with open(progress_path, "w") as f:
        json.dump(report, f, indent=2)


started_at = time.time()
for step in range(start_step, total_steps):
    x, y = make_training_batch()
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1),
                           ignore_index=-1)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    is_report_step = (step + 1) % 250 == 0 or step + 1 == total_steps
    if is_report_step or step == start_step:
        vloss = evaluate_val_loss(model)
        save_checkpoint_and_progress(step + 1, loss.item(), vloss)
        minutes = (time.time() - started_at) / 60
        print("step %5d / %d | train loss %.3f | val loss %.3f | %.1f min"
              % (step + 1, total_steps, loss.item(), vloss, minutes))

if start_step >= total_steps:
    print("this shard already finished in an earlier session, nothing left to train")

print()
print("DONE. Your shard is green. Checkpoint and progress file are in:")
print("  " + TEAM_FOLDER)
print("Now run the next cell, then go help any teammate who is still red.")

In [ ]:
# ===========================================================================
# TEAM STATUS: who is green? Run this anytime, it reads every progress file
# in the shared folder. It also downloads YOUR progress JSON so you can drop
# it into the team panel on the lab page (works from file:// too).
# ===========================================================================
import glob
import json

my_progress_path = TEAM_FOLDER + "/progress-member-" + str(MEMBER_NUMBER) + ".json"

progress_files = sorted(glob.glob(TEAM_FOLDER + "/progress-member-*.json"))
if len(progress_files) == 0:
    print("no progress files yet, run the training cell first")

print("%-8s %-12s %-14s %-10s %s" % ("member", "name", "step", "val loss", "status"))
green_count = 0
for path in progress_files:
    with open(path) as f:
        report = json.load(f)
    if report["done"]:
        status = "GREEN, ready to merge"
        green_count = green_count + 1
    else:
        status = "training (last save " + str(report.get("updated", "?")) + ")"
    steps_text = str(report["step"]) + " / " + str(report["total_steps"])
    print("%-8s %-12s %-14s %-10s %s" % (report["member"], report["name"],
                                         steps_text, report["val_loss"], status))

print()
print(green_count, "of 4 shards green")
if green_count >= 4:
    print("ALL GREEN: any one teammate can run the merge cell below.")

# Download your own progress file for the lab page team panel.
try:
    from google.colab import files
    files.download(my_progress_path)
except Exception:
    print("(not on Colab, skipping the browser download)")

## The merge: four quarter-models become one

Each checkpoint in the shared folder is a full copy of the model, but each one only ever saw its own quarter of the names. The merge is almost embarrassingly simple: **for every weight in the network, take the average of the four teammates' values.**

Why does averaging work instead of producing mush? Because all four models started from the SAME random weights (seed 42) and only trained for a short while, they stayed in the same "valley" of the loss landscape. Averaging four nearby points in the same valley gives a point near the middle, which cancels out each model's shard-specific quirks while keeping what they all agree on. This is federated averaging (McMahan et al., 2017), and it is how models get trained across hospitals or phones without the raw data ever leaving home.

Any ONE teammate runs the next cell (it is fine if everyone runs it, the result is identical).

In [ ]:
# ===========================================================================
# MERGE AND EVAL: average the weights, then prove the merged model wins.
# ===========================================================================

# ---- 1. load every member checkpoint we can find --------------------------
member_states = {}
for m in [1, 2, 3, 4]:
    path = TEAM_FOLDER + "/member-" + str(m) + ".pt"
    if os.path.exists(path):
        saved = torch.load(path, map_location=device)
        member_states[m] = saved["model"]

found = sorted(member_states.keys())
missing = []
for m in [1, 2, 3, 4]:
    if m not in member_states:
        missing.append(m)

print("checkpoints found for members:", found)
if len(missing) > 0:
    print("still missing:", missing, "(a full table is 4, go help them!)")
if len(found) < 2:
    raise SystemExit("need at least 2 checkpoints to merge")

# ---- 2. federated averaging: mean of every weight tensor ------------------
merged_state = {}
for key in member_states[found[0]]:
    total = member_states[found[0]][key].clone().float()
    for m in found[1:]:
        total = total + member_states[m][key].float()
    merged_state[key] = total / len(found)

merged_model = TinyGPT().to(device)
merged_model.load_state_dict(merged_state)
torch.save({"model": merged_state, "step": total_steps},
           TEAM_FOLDER + "/merged.pt")
print("merged", len(found), "checkpoints, saved merged.pt to the team folder")
print()

# ---- 3. the knowledge map: loss per alphabet quarter ------------------------
# Overall val loss says who wins. The per-quarter columns say WHY: every solo
# model is good at exactly the letters it trained on and lost everywhere
# else. The merged model is strong across the whole alphabet.
val_by_quarter = {}
for q in [1, 2, 3, 4]:
    val_by_quarter[q] = []
for name in val_names:
    for q in [1, 2, 3, 4]:
        if name[0] in quarter_letters[q]:
            val_by_quarter[q].append(name)


def quarter_losses(the_model):
    the_model.eval()
    row = []
    with torch.no_grad():
        for q in [1, 2, 3, 4]:
            loss = loss_on_names(the_model, val_by_quarter[q][:400])
            row.append(loss.item())
    return row


header = "%-16s" % "model"
for q in [1, 2, 3, 4]:
    span = quarter_letters[q][0] + "-" + quarter_letters[q][-1]
    header = header + "%9s" % span
header = header + "%10s %9s" % ("OVERALL", "worst")
print(header)

solo_summaries = []
best_solo_overall = None
best_solo_model = None
for m in found:
    solo_model = TinyGPT().to(device)
    solo_model.load_state_dict(member_states[m])
    row = quarter_losses(solo_model)
    overall = evaluate_val_loss(solo_model)
    worst = max(row)
    line = "%-16s" % ("member " + str(m) + " solo")
    for v in row:
        line = line + "%9.3f" % v
    print(line + "%10.4f %9.3f" % (overall, worst))
    solo_summaries.append({"member": m,
                           "val_loss": round(overall, 4),
                           "worst_quarter_loss": round(worst, 4)})
    if best_solo_overall is None or overall < best_solo_overall:
        best_solo_overall = overall
        best_solo_model = solo_model

merged_row = quarter_losses(merged_model)
merged_overall = evaluate_val_loss(merged_model)
merged_worst = max(merged_row)
line = "%-16s" % "MERGED"
for v in merged_row:
    line = line + "%9.3f" % v
print(line + "%10.4f %9.3f" % (merged_overall, merged_worst))

# ---- 4. the verdict ---------------------------------------------------------
best_solo_worst = None
for s in solo_summaries:
    if best_solo_worst is None or s["worst_quarter_loss"] < best_solo_worst:
        best_solo_worst = s["worst_quarter_loss"]

print()
if merged_overall < best_solo_overall:
    print("LOSS: merged wins. %.4f vs %.4f for the best solo model,"
          % (merged_overall, best_solo_overall))
    print("      at about a quarter of the wall-clock of a solo full-data run.")
else:
    print("Unusual: merged did not beat the best solo overall this run.")
    print("Check that everyone ran the same seed cell and finished training.")
if merged_worst < best_solo_worst:
    print("CHAT QUALITY: merged wins. Its weakest alphabet quarter (%.3f) is"
          % merged_worst)
    print("      still better than any solo model's weakest quarter (%.3f)."
          % best_solo_worst)
    print("      It is the only model on the table you can ask ANYTHING.")

# ---- 5. taste test: ask every model about every quarter --------------------
# Two held-out names from each quarter. Prompt = the first 3 letters, and
# each model answers with its single most likely completion (greedy decode,
# no dice), so differences between models are real, not sampling luck.
# Watch each solo model do fine on its own letters and improvise everywhere
# else, while the merged model stays plausible across the board.

def greedy_name(the_model, prefix):
    the_model.eval()
    ids = [BOS]
    for ch in prefix:
        ids.append(stoi[ch])
    with torch.no_grad():
        while len(ids) < block_size:
            x = torch.tensor([ids], dtype=torch.long, device=device)
            logits = the_model(x)
            next_id = int(torch.argmax(logits[0, -1]).item())
            if next_id == BOS:
                break
            ids.append(next_id)
    name = ""
    for token in ids[1:]:
        name = name + itos[token]
    return name

print()
prompts = []
for q in [1, 2, 3, 4]:
    prompts.append(val_by_quarter[q][0][:3])
    prompts.append(val_by_quarter[q][1][:3])

taste_models = []
for m in found:
    solo_model = TinyGPT().to(device)
    solo_model.load_state_dict(member_states[m])
    taste_models.append(("m" + str(m), solo_model))
taste_models.append(("MERGED", merged_model))

head = "%-8s" % "prompt"
for tag, _ in taste_models:
    head = head + "%-14s" % tag
print(head + "(each model's most likely completion of a held-out name)")
merged_taste = []
for prompt in prompts:
    line = "%-8s" % (prompt + "..")
    for tag, tm in taste_models:
        got = greedy_name(tm, prompt)
        if tag == "MERGED":
            merged_taste.append(got)
        line = line + "%-14s" % got
    print(line)

In [ ]:
# ===========================================================================
# CHAT WITH THE TEAM MODEL. Every teammate can run this, it loads merged.pt
# from the shared folder. Type letter-prefixes, get names, exactly like the
# Part 2 bot (one name per whitespace-separated prefix).
# ===========================================================================
merged_ckpt = torch.load(TEAM_FOLDER + "/merged.pt", map_location=device)
team_model = TinyGPT().to(device)
team_model.load_state_dict(merged_ckpt["model"])


def team_chat(request):
    answers = []
    for raw in request.strip().lower().split():
        prefix = ""
        for ch in raw:
            if ch in stoi:
                prefix = prefix + ch
        got = sample_names(team_model, 1, prefix)
        answers.append(got[0])
    print("you:  " + request)
    print("bot:  " + ", ".join(answers))
    print()


team_chat("a bri jo z")
team_chat("k k k")
# Your turn: team_chat("<your prefixes here>")

In [ ]:
# ===========================================================================
# TEAM SCORECARD: one small JSON for the cross-team leaderboard on the lab
# page. Run after the merge cell. Post it in the class chat, or hand every
# team's file to the projector laptop and load them all in the leaderboard.
import json
import time

minutes_spent = round((time.time() - started_at) / 60, 1)

team_result = {"kind": "tiny-ai-team-result",
               "team": TEAM_NAME,
               "members": found,
               "solo": solo_summaries,
               "merged_val_loss": round(merged_overall, 4),
               "merged_worst_quarter_loss": round(merged_worst, 4),
               "best_solo_val_loss": round(best_solo_overall, 4),
               "samples": merged_taste,
               "train_minutes": minutes_spent}

result_path = TEAM_FOLDER + "/team-result.json"
with open(result_path, "w") as f:
    json.dump(team_result, f, indent=2)

print(json.dumps(team_result, indent=2))
try:
    from google.colab import files
    files.download(result_path)
except Exception:
    print("(not on Colab, skipping the browser download)")